In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

import math

from attention import causal_mask

In [ ]:
class MultiAttention(nn.Module):
    def __init__(self, embed_dim, qk_dim, v_dim, head_size, dropout=0):
        super().__init__()

        self.q = nn.Linear(embed_dim, head_size * qk_dim)
        self.k = nn.Linear(embed_dim, head_size * qk_dim)
        self.v = nn.Linear(embed_dim, head_size * v_dim)
        self.proj = nn.Linear(head_size * v_dim, embed_dim)
        self.qk_dim = qk_dim
        self.v_dim = v_dim

        self.dropout = nn.Dropout(dropout)
        self.head = head_size

        self.apply(self._init_weights)

    def forward(self, x):
        # x 的维度是 (B, seq_len,)

        # 有必要先变成 (B, Seq_len, head_size, qk_dim),再变成(B, head_size, Seq_len, qk_dim)
        # 保证每一个头的信息都是自己的

        # (B, head_size, Seq_len, qk_dim)
        Q = self.q(x).view(x.shape[0], x.shape[1], self.head, self.qk_dim).transpose(1, 2)
        K = self.k(x).view(x.shape[0], x.shape[1], self.head, self.qk_dim).transpose(1, 2)

        print(f"Q 矩阵：")
        print(Q)

        # (B, head_size, Seq_len, v_dim)
        V = self.v(x).view(x.shape[0], x.shape[1], self.head, self.v_dim).transpose(1, 2)

        print(f"V 矩阵：")
        print(V)

        # .mT 矩阵转置
        # (B, head_size, Seq_len, Seq_len)
        raw_scores = (Q @ K.mT) / math.sqrt(Q.shape[-1])

        # 因果掩码
        raw_scores = causal_mask(raw_scores)

        # (B, head_size, seq_len, seq_len)
        scores = self.dropout(F.softmax(raw_scores, dim=-1))

        # (B, head_size, seq_len, v_dim)
        res = scores @ V

        # (B, seq_len, head_size * v_dim)
        res = res.transpose(1, 2).reshape(res.shape[0], x.shape[1], -1)

        # (B, seq_len, embed_dim)
        res = self.proj(res)

        return res

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.ones_(module.weight)

            if module.bias is not None:
                nn.init.zeros_(module.bias)